In [10]:
import torch
import torch.nn.functional as F

In [11]:
words = open('names.txt').read().splitlines()
words[:3]

['emma', 'olivia', 'ava']

In [12]:
unique = sorted(list(set(''.join(words))))
unique[:3]

['a', 'b', 'c']

In [13]:
c_to_idx = {}
idx_to_c = {}

for idx, c in enumerate(unique):
    c_to_idx[c] = idx + 1
    idx_to_c[idx + 1] = c

c_to_idx['.'] = 0
idx_to_c[0] = '.'

In [14]:
print(c_to_idx)

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}


In [15]:
print(idx_to_c)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [16]:
x = []
y = []

for word in words[:3]:
    sep_word = list('.') + list(word) + list('.')

    for ch1, ch2 in zip(sep_word, sep_word[1:]):
        idx1 = c_to_idx[ch1]
        idx2 = c_to_idx[ch2]

        x.append(idx1)
        y.append(idx2)

x = torch.tensor(x)
y = torch.tensor(y)

print(x)
print(y)

tensor([ 0,  5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1])
tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0])


In [17]:
# input matrix of size (b, 27)

enc_x = F.one_hot(x, num_classes=27).float()
print(enc_x.dtype)
enc_x.shape

torch.float32


torch.Size([16, 27])

In [18]:
# weight matrix 

w = torch.randn((27, 27), requires_grad=True) # gives a weight matrix initialized in the normal distribution

print(w.dtype)
w.shape

torch.float32


torch.Size([27, 27])

In [19]:
logits = enc_x @ w
# (10, 27) * (27, 27) => (10, 27)
logits.shape

torch.Size([16, 27])

In [20]:
exp = logits.exp()
print(exp.shape)
exp

torch.Size([16, 27])


tensor([[ 1.9213,  2.4165,  3.3353,  2.0575,  1.1855,  0.4850,  1.2693,  1.8325,
          0.3307,  1.5493,  0.9116,  0.2648,  0.6631,  1.3728,  0.5399,  1.2258,
          0.3520,  0.9433,  0.1446,  1.8832,  2.5091,  1.2121,  1.0506,  0.9415,
          5.5496,  6.1231,  1.0929],
        [ 3.0813,  0.7147,  1.6692,  0.4322,  0.8848,  1.7759,  0.8856,  0.5875,
          4.5504,  0.3278,  0.7631,  0.8984,  4.9098,  1.2036,  0.8732,  1.5258,
          0.5883,  0.1706,  3.1919,  0.9066,  1.4548,  1.3568,  0.1117,  1.5448,
          0.2232,  1.2035,  0.6447],
        [ 0.6803,  0.5429,  0.1084,  0.5520,  5.1144,  0.9038,  0.8850,  1.2602,
          2.0866,  2.3423,  1.1125,  0.6705,  0.9044,  1.9436,  0.4763,  0.3142,
          8.0973,  2.1278,  1.7090,  0.5559,  2.7744,  1.5053,  1.0981,  0.4921,
          1.8499,  0.0335,  1.0940],
        [ 0.6803,  0.5429,  0.1084,  0.5520,  5.1144,  0.9038,  0.8850,  1.2602,
          2.0866,  2.3423,  1.1125,  0.6705,  0.9044,  1.9436,  0.4763,  0.3142

In [21]:
test = exp.sum(dim=1)
print(test.shape)
test # tensor with sum of each row (each input) that is given to the neural network

torch.Size([16])


tensor([43.1631, 36.4805, 41.2349, 41.2349, 42.1549, 43.1631, 30.5774, 44.7505,
        38.3045, 62.6508, 38.3045, 42.1549, 43.1631, 42.1549, 62.6508, 42.1549],
       grad_fn=<SumBackward1>)

In [22]:
counts = exp.sum(dim=1, keepdim=True)
print(counts.shape)
counts

torch.Size([16, 1])


tensor([[43.1631],
        [36.4805],
        [41.2349],
        [41.2349],
        [42.1549],
        [43.1631],
        [30.5774],
        [44.7505],
        [38.3045],
        [62.6508],
        [38.3045],
        [42.1549],
        [43.1631],
        [42.1549],
        [62.6508],
        [42.1549]], grad_fn=<SumBackward1>)

In [23]:
probs = exp / counts # this is brodcastible and hence works
probs

tensor([[0.0445, 0.0560, 0.0773, 0.0477, 0.0275, 0.0112, 0.0294, 0.0425, 0.0077,
         0.0359, 0.0211, 0.0061, 0.0154, 0.0318, 0.0125, 0.0284, 0.0082, 0.0219,
         0.0034, 0.0436, 0.0581, 0.0281, 0.0243, 0.0218, 0.1286, 0.1419, 0.0253],
        [0.0845, 0.0196, 0.0458, 0.0118, 0.0243, 0.0487, 0.0243, 0.0161, 0.1247,
         0.0090, 0.0209, 0.0246, 0.1346, 0.0330, 0.0239, 0.0418, 0.0161, 0.0047,
         0.0875, 0.0249, 0.0399, 0.0372, 0.0031, 0.0423, 0.0061, 0.0330, 0.0177],
        [0.0165, 0.0132, 0.0026, 0.0134, 0.1240, 0.0219, 0.0215, 0.0306, 0.0506,
         0.0568, 0.0270, 0.0163, 0.0219, 0.0471, 0.0116, 0.0076, 0.1964, 0.0516,
         0.0414, 0.0135, 0.0673, 0.0365, 0.0266, 0.0119, 0.0449, 0.0008, 0.0265],
        [0.0165, 0.0132, 0.0026, 0.0134, 0.1240, 0.0219, 0.0215, 0.0306, 0.0506,
         0.0568, 0.0270, 0.0163, 0.0219, 0.0471, 0.0116, 0.0076, 0.1964, 0.0516,
         0.0414, 0.0135, 0.0673, 0.0365, 0.0266, 0.0119, 0.0449, 0.0008, 0.0265],
        [0.0295, 0.0179,

In [24]:
sum(probs[1])

tensor(1.0000, grad_fn=<AddBackward0>)

In [25]:
# calculating loss

nll = torch.zeros(len(y)) # stores the negative log loss for each input
print(len(y))
for i in range(len(y)): # for each output we will get a loss

    prob = probs[i, y[i]]
    log_loss = torch.log(prob)
    neg_log_loss = -log_loss
    nll[i] = neg_log_loss

loss = nll.mean()
loss

16


tensor(3.6600, grad_fn=<MeanBackward0>)

In [26]:
# backward prop

w.grad = None # set gradients to zero
w.grad

loss.backward()

In [27]:
w.data += -0.1 * w.grad

In [28]:
# ok now setting all of this up in clean cells 

words = open('names.txt').read().splitlines()

unique = sorted(list(set(''.join(words))))

c_to_idx = {}
idx_to_c = {}

for idx, c in enumerate(unique):
    c_to_idx[c] = idx + 1
    idx_to_c[idx + 1] = c

c_to_idx['.'] = 0
idx_to_c[0] = '.'

x = []
y = []

for word in words:
    sep_word = list('.') + list(word) + list('.')

    for ch1, ch2 in zip(sep_word, sep_word[1:]):
        idx1 = c_to_idx[ch1]
        idx2 = c_to_idx[ch2]

        x.append(idx1)
        y.append(idx2)

x = torch.tensor(x)
y = torch.tensor(y)

print(x)
print(y)

tensor([ 0,  5, 13,  ..., 25, 26, 24])
tensor([ 5, 13, 13,  ..., 26, 24,  0])


In [29]:
# input matrix of size (b, 27)

enc_x = F.one_hot(x, num_classes=27).float()
print(enc_x.dtype)

# generator
g = torch.Generator().manual_seed(2147483647)

# weight matrix 
w = torch.randn((27, 27), generator=g, requires_grad=True) # gives a weight matrix initialized in the normal distribution


torch.float32


In [30]:
# gradient descent

epochs = 100
lr = 50

for i in range(epochs):

    # forward prop

    logits = enc_x @ w
    # (10, 27) * (27, 27) => (10, 27)
    
    # softmax
    exp = logits.exp()
    counts = exp.sum(dim=1, keepdim=True)
    probs = exp / counts # this is brodcastible and hence works

    # calculating loss
    nll = torch.zeros(len(y)) # stores the negative log loss for each input

    '''
    for i in range(len(y)): # for each output we will get a loss

        prob = probs[i, y[i]]
        log_loss = torch.log(prob)
        neg_log_loss = -log_loss
        nll[i] = neg_log_loss

    loss = nll.mean()
    very slow and can be done using one vectorized PyTorch operation
    '''
    
    loss = -probs[torch.arange(len(y)), y].log().mean()

    print(loss.item())

    # backward prop
    w.grad = None # setting gradients to zero
    loss.backward()

    w.data += -lr * w.grad
    
    

3.758953809738159
3.371100902557373
3.154043197631836
3.020373821258545
2.927711248397827
2.8604023456573486
2.8097290992736816
2.7701022624969482
2.7380728721618652
2.711496591567993
2.6890032291412354
2.6696882247924805
2.6529300212860107
2.638277769088745
2.6253881454467773
2.6139907836914062
2.60386323928833
2.5948219299316406
2.5867116451263428
2.5794036388397217
2.572789192199707
2.5667762756347656
2.5612878799438477
2.5562586784362793
2.551633596420288
2.547365665435791
2.5434155464172363
2.5397486686706543
2.5363364219665527
2.533154249191284
2.5301806926727295
2.5273966789245605
2.5247862339019775
2.522334575653076
2.520029067993164
2.5178580284118652
2.515810489654541
2.513878345489502
2.512052059173584
2.510324001312256
2.5086867809295654
2.5071346759796143
2.5056614875793457
2.5042612552642822
2.5029289722442627
2.5016608238220215
2.5004522800445557
2.4992988109588623
2.498197317123413
2.497144937515259
2.4961376190185547
2.495173692703247
2.4942493438720703
2.4933633804321

In [31]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ w # predict log-counts
        counts = logits.exp() # equivalent to N matrix
        p = counts / counts.sum(1, keepdim=True)

        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(idx_to_c[ix])
        if ix == 0:
            break
    print(''.join(out))

junide.
janasah.
p.
cfay.
a.
